# 专利数据正式清洗

清洗逻辑复用 src.patent_cleaning，不修改 raw，并输出 Parquet 与 Stata。

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, '..')
import pandas as pd

from src.patent_cleaning import clean_patents, stata_ready

raw_path = Path('../data/raw/patents.csv')
output_dir = Path('../data/processed')
raw = pd.read_csv(raw_path, dtype=object, keep_default_na=False)
clean = clean_patents(raw)
clean.to_parquet(output_dir / 'patents_clean.parquet', index=False)
stata_ready(clean).to_stata(output_dir / 'patents_clean.dta', write_index=False, version=118)
clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 220 entries, 0 to 219
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   stock_code         220 non-null    string
 1   year               220 non-null    Int64 
 2   invention_patents  219 non-null    Int64 
 3   utility_patents    219 non-null    Int64 
 4   patent_citations   218 non-null    Int64 
dtypes: Int64(4), string(1)
memory usage: 10.9 KB


In [2]:
assert len(clean) == 220
assert not clean.duplicated(['stock_code', 'year']).any()
assert clean['stock_code'].str.fullmatch(r'[0-9]{6}').all()
assert clean['year'].between(2020, 2025).all()
numeric_columns = ['invention_patents', 'utility_patents', 'patent_citations']
assert (clean[numeric_columns].dropna() >= 0).all().all()
pd.read_parquet(output_dir / 'patents_clean.parquet').head()

,stock_code,year,invention_patents,utility_patents,patent_citations
0,000001,2020,8,19,33
1,000001,2021,8,16,23
2,000001,2023,8,9,24
3,000001,2024,10,8,29
4,000001,2025,12,7,39


## 清洗规则记录

股票代码执行 strip、纯数字验证和 zfill(6)；年份转为 nullable integer 并限定在 2020–2025；三个数值字段转为 nullable integer，保留真实缺失且拒绝负数和非整数。完全重复只保留一份；冲突重复先对明显稳健分布异常的引用数设为 missing，再保留缺失更少、证据更充分的版本。